# Entitiy Resolution PoC

Dieser PoC beschäftigt sich mit Entity Resolution (ER) mit Matryoshka Embeddings von `jina-embeddings-v4` und Hierarchical Navigable Small World (HNSW) für die ANN Suche als zu grunde liegende Technologien. Ziel ist die Maximierung der Konsistenz von oft unsauberen Daten.

## Setup

In [ ]:
# ── 1. Environment Detection ──────────────────────────────────────────────────
import subprocess
import sys
import os

def in_kaggle():
    return "KAGGLE_KERNEL_RUN_TYPE" in os.environ

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["transformers", "faker", "matplotlib", "seaborn", "scikit-learn", "pandas", "numpy"]:
    install(pkg)

try:
    import torch
except ImportError:
    install("torch")

try:
    import faiss
except ImportError:
    if in_kaggle():
        subprocess.check_call(["conda", "install", "-y", "-c", "pytorch", "-c", "nvidia", "faiss-gpu"])
    else:
        install("faiss-cpu")  # Mac M4 → faiss-cpu

# ── 2. Device Setup ───────────────────────────────────────────────────────────
import torch

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")       # Kaggle T4
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")        # Mac M4
else:
    DEVICE = torch.device("cpu")

print(f"Environment : {'Kaggle' if in_kaggle() else 'Local'}")
print(f"Device      : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU         : {torch.cuda.get_device_name(0)}")

# ── 3. FAISS Index Helper (GPU-aware) ─────────────────────────────────────────
import faiss

def make_faiss_index(dim: int) -> faiss.Index:
    """Gibt GPU-Index auf Kaggle zurück, CPU-Index sonst."""
    index = faiss.IndexFlatL2(dim)
    if DEVICE.type == "cuda":
        res = faiss.StandardGpuResources()
        index = faiss.index_cpu_to_gpu(res, 0, index)
    return index

Environment : Local
Device      : mps
